# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook demonstrates parsing synthetic raw EDI lines and Debezium JSON records into structured Bronze output.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, expr, col, from_json
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
# 1. X12 Batch Parsing Simulation
x12_data = ["ISA*00*          *00*          *ZZ*1234567890     *ZZ*0987654321     *230501*1200*^*00501*000000001*0*T*:~\nGS*HC*1234567890*0987654321*20230501*1200*1*X*005010X222A1~\nST*837*0001*005010X222A1~"]
df_x12_raw = spark.createDataFrame([(line,) for line in x12_data], ["value"])

df_x12_parsed = df_x12_raw.selectExpr("concat('{\\\"raw_edi\\\": \\\"', value, '\\\"}') as raw_payload_json") \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("x12_batch_sim")) \
    .withColumn("payload_id", expr("uuid()"))

print("Parsed X12 Batch Data:")
df_x12_parsed.show(truncate=False)

In [ ]:
# 2. Debezium CDC Simulation
cdc_json_data = [
    '{"payload": {"op": "c", "after": "{\"id\": 1, \"name\": \"John Doe\"}"}}',
    '{"payload": {"op": "u", "after": "{\"id\": 1, \"name\": \"John Doe Updated\"}"}}',
    '{"payload": {"op": "r", "after": "{\"id\": 2, \"name\": \"Jane Doe\"}"}}', # Should be filtered out
]
df_cdc_raw = spark.createDataFrame([(line,) for line in cdc_json_data], ["json_val"])

debezium_schema = StructType([
    StructField("payload", StructType([
        StructField("op", StringType(), True),
        StructField("after", StringType(), True)
    ]))
])

df_cdc_parsed = df_cdc_raw \
    .withColumn("data", from_json(col("json_val"), debezium_schema)) \
    .select(col("data.payload.op").alias("_cdc_op"), col("data.payload.after").alias("raw_payload_json")) \
    .filter(col("_cdc_op").isin("c", "u", "d")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("debezium_cdc_sim")) \
    .withColumn("payload_id", expr("uuid()"))

print("Processed CDC Data:")
df_cdc_parsed.show(truncate=False)